# CSL7110 – Machine Learning with Big Data  
## Assignment 3 – Movie Recommendation Systems

**Name:** Ruby Mythili  
**Roll Number:** M25DE1006  
**Course:** CSL7110 – Machine Learning with Big Data  
**Platform:** Google Colab

## Objective

The goal of this assignment is to implement and compare different recommendation system techniques using the MovieLens dataset.  
The methods implemented include:

1. Content-Based Filtering  
2. Collaborative Filtering  
3. Matrix Factorization  
4. Hybrid Recommendation Systems  
5. Learning-Based Recommendation Models  
6. Explainability in Recommendation Systems  

The objective is to analyze how different recommendation algorithms perform and understand their strengths and limitations.

## Dataset

This assignment uses the **MovieLens Small Dataset**, which contains:

- Movie information
- User ratings
- Movie tags
- External links

The dataset is widely used for experimentation with recommender systems.

## Assumptions

- The **MovieLens small dataset** is used for experimentation.
- Ratings **greater than or equal to 4** are considered positive preferences.
- Genre information is used for content-based filtering.
- Evaluation metrics include **RMSE, Precision@K, and Recall@K**.
- Since real-time user feedback is unavailable, reinforcement learning rewards are simulated using rating values.

In [1]:
# ============================================================
# ASSIGNMENT 3 - SETUP
# Installing the required libraries for recommendation system tasks
# ============================================================

!pip uninstall -y numpy surprise scikit-surprise
!pip install "numpy<2"
!pip install scikit-surprise==1.1.4
!pip install lime

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python

  Using cached scikit_surprise-1.1.4-cp312-cp312-linux_x86_64.whl


In [1]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# This cell imports all libraries needed for data loading,
# preprocessing, recommendation models, and evaluation
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split as surprise_split
from surprise import accuracy

import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully")

All libraries imported successfully


In [3]:
# ============================================================
# DATASET DOWNLOAD
# Downloading the MovieLens small dataset used for
# building recommendation systems
# ============================================================

!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip ml-latest-small.zip

--2026-03-08 17:46:20--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip.1’

ml-latest-small.zip 100%[===================>] 955.28K   905KB/s    in 1.1s    

2026-03-08 17:46:23 (905 KB/s) - ‘ml-latest-small.zip.1’ saved [978202/978202]

Archive:  ml-latest-small.zip
replace ml-latest-small/links.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [2]:
# ============================================================
# DATA LOADING
# Loading MovieLens dataset files into pandas DataFrames
# ============================================================

movies = pd.read_csv("ml-latest-small/movies.csv")
ratings = pd.read_csv("ml-latest-small/ratings.csv")
tags = pd.read_csv("ml-latest-small/tags.csv")
links = pd.read_csv("ml-latest-small/links.csv")

print("Movies dataset shape:", movies.shape)
print("Ratings dataset shape:", ratings.shape)
print("Tags dataset shape:", tags.shape)
print("Links dataset shape:", links.shape)

Movies dataset shape: (9742, 3)
Ratings dataset shape: (100836, 4)
Tags dataset shape: (3683, 4)
Links dataset shape: (9742, 3)


In [3]:
# ============================================================
# DATA UNDERSTANDING
# Question: Inspect the structure of the MovieLens dataset
# Display first few rows of movies and ratings tables
# ============================================================

print("Sample movies dataset:")
display(movies.head())

print("\nSample ratings dataset:")
display(ratings.head())

Sample movies dataset:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



Sample ratings dataset:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [5]:
# ============================================================
# Task 1: Content-Based Filtering using TF-IDF
#
# Objective:
# Build a content-based recommendation system using movie genres.
# We convert genres into TF-IDF vectors and compute similarity
# between movies using cosine similarity.
# ============================================================

# Replace '|' with space so TF-IDF treats genres as words
movies['genres'] = movies['genres'].str.replace('|', ' ', regex=False)

# Create TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words='english')

# Transform genres into TF-IDF matrix
tfidf_matrix = tfidf.fit_transform(movies['genres'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (9742, 23)


In [6]:
# ============================================================
# Task 1: Content-Based Filtering
#
# Step 2:
# Compute cosine similarity between all movies based on TF-IDF
# genre representation
# ============================================================

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine similarity matrix shape:", cosine_sim.shape)

Cosine similarity matrix shape: (9742, 9742)


In [7]:
# ============================================================
# Task 1: Content-Based Filtering
#
# Step 3:
# Create a mapping between movie titles and index positions
# in the dataset. This helps us quickly retrieve similarity
# scores for a given movie.
# ============================================================

indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

print("Index mapping created successfully")

Index mapping created successfully


In [8]:
# ============================================================
# Task 1: Content-Based Filtering
#
# Step 4:
# Define a function to recommend top similar movies
# based on cosine similarity of genre TF-IDF vectors
# ============================================================

def get_content_recommendations(title, cosine_sim=cosine_sim):
    # Get index of the movie that matches the title
    idx = indices[title]

    # Get pairwise similarity scores for all movies with the selected movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort movies based on similarity score in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Ignore the first movie because it is the same movie itself
    sim_scores = sim_scores[1:11]

    # Get movie indices
    movie_indices = [i[0] for i in sim_scores]

    # Return top 10 recommended movie titles
    return movies[['title', 'genres']].iloc[movie_indices]

print("Content-based recommendation function created successfully")

Content-based recommendation function created successfully


In [9]:
# ============================================================
# Task 1: Content-Based Filtering
#
# Step 5:
# Test the recommendation system for a sample movie
# ============================================================

print("Top 10 movies similar to Toy Story (1995):")
display(get_content_recommendations("Toy Story (1995)"))

Top 10 movies similar to Toy Story (1995):


,title,genres
1706,Antz (1998),Adventure Animation Children Comedy Fantasy
2355,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure Animation Children Comedy Fantasy
3000,"Emperor's New Groove, The (2000)",Adventure Animation Children Comedy Fantasy
3568,"Monsters, Inc. (2001)",Adventure Animation Children Comedy Fantasy
6194,"Wild, The (2006)",Adventure Animation Children Comedy Fantasy
6486,Shrek the Third (2007),Adventure Animation Children Comedy Fantasy
6948,"Tale of Despereaux, The (2008)",Adventure Animation Children Comedy Fantasy
7760,Asterix and the Vikings (Astérix et les Viking...,Adventure Animation Children Comedy Fantasy
8219,Turbo (2013),Adventure Animation Children Comedy Fantasy


In [10]:
# ============================================================
# Task 1: Content-Based Filtering
#
# Step 6:
# Build a user profile using movies previously rated by a user.
# This helps recommend movies based on the user's preferences.
# ============================================================

# Merge ratings with movie information
data = pd.merge(ratings, movies, on='movieId')

# Select a sample user
user_id = 1

# Get movies rated by this user
user_movies = data[data['userId'] == user_id]

print("Number of movies rated by the user:", len(user_movies))

# Display some rated movies
user_movies[['title', 'rating']].head()

Number of movies rated by the user: 232


,title,rating
0,Toy Story (1995),4.0
1,Grumpier Old Men (1995),4.0
2,Heat (1995),4.0
3,Seven (a.k.a. Se7en) (1995),5.0
4,"Usual Suspects, The (1995)",5.0


In [11]:
# ============================================================
# Task 1: Content-Based Filtering
#
# Step 7:
# Build a user preference vector by taking the weighted average
# of genre TF-IDF vectors using the user's ratings.
# ============================================================

# Get indices of movies rated by the selected user
user_movie_indices = user_movies['movieId'].map(
    pd.Series(movies.index, index=movies['movieId'])
).dropna().astype(int).values

# Get the corresponding ratings
user_ratings = user_movies['rating'].values

# Extract TF-IDF vectors for the movies rated by the user
user_tfidf = tfidf_matrix[user_movie_indices]

# Compute weighted average user profile
user_profile = np.average(user_tfidf.toarray(), axis=0, weights=user_ratings)

print("User profile vector shape:", user_profile.shape)

User profile vector shape: (23,)


In [12]:
# ============================================================
# Task 1: Content-Based Filtering
#
# Step 8:
# Compute similarity between the user profile and all movies
# to generate personalized recommendations
# ============================================================

# Compute similarity between user profile and all movie vectors
scores = cosine_similarity([user_profile], tfidf_matrix)

print("Similarity score matrix shape:", scores.shape)

Similarity score matrix shape: (1, 9742)


In [14]:
# ============================================================
# Task 1: Content-Based Filtering
#
# Step 9:
# Recommend top movies for the user based on similarity scores
# ============================================================

# Convert scores to a flat array
scores = scores.flatten()

# Get indices of top recommended movies
top_indices = scores.argsort()[-10:][::-1]

# Display recommended movies
recommended_movies = movies.iloc[top_indices][['title','genres']]

print("Top recommended movies for the user:")
recommended_movies

Top recommended movies for the user:


,title,genres
8597,Dragonheart 2: A New Beginning (2000),Action Adventure Comedy Drama Fantasy Thriller
6570,"Hunting Party, The (2007)",Action Adventure Comedy Drama Thriller
4005,Flashback (1990),Action Adventure Comedy Crime Drama
4681,The Great Train Robbery (1978),Action Adventure Comedy Crime Drama
4409,Charlie's Angels: Full Throttle (2003),Action Adventure Comedy Crime Thriller
5379,After the Sunset (2004),Action Adventure Comedy Crime Thriller
5471,"Diamond Arm, The (Brilliantovaya ruka) (1968)",Action Adventure Comedy Crime Thriller
7409,Machete (2010),Action Adventure Comedy Crime Thriller
9394,Maximum Ride (2016),Action Adventure Comedy Fantasy Sci-Fi Thriller
3608,"Stunt Man, The (1980)",Action Adventure Comedy Drama Romance Thriller


In [15]:
# ============================================================
# Task 1: Content-Based Filtering
#
# Step 10:
# Remove movies already rated by the user and generate
# top unseen personalized recommendations
# ============================================================

# Get movieIds already rated by the user
rated_movie_ids = set(user_movies['movieId'])

# Add similarity scores to movies dataframe
movies['similarity_score'] = scores

# Remove already rated movies
unseen_movies = movies[~movies['movieId'].isin(rated_movie_ids)]

# Get top 10 unseen recommendations
top_unseen_recommendations = unseen_movies.sort_values(
    by='similarity_score', ascending=False
)[['title', 'genres', 'similarity_score']].head(10)

print("Top unseen recommended movies for the user:")
top_unseen_recommendations

Top unseen recommended movies for the user:


,title,genres,similarity_score
8597,Dragonheart 2: A New Beginning (2000),Action Adventure Comedy Drama Fantasy Thriller,0.791026
6570,"Hunting Party, The (2007)",Action Adventure Comedy Drama Thriller,0.773645
4005,Flashback (1990),Action Adventure Comedy Crime Drama,0.773089
4681,The Great Train Robbery (1978),Action Adventure Comedy Crime Drama,0.773089
5471,"Diamond Arm, The (Brilliantovaya ruka) (1968)",Action Adventure Comedy Crime Thriller,0.760503
5379,After the Sunset (2004),Action Adventure Comedy Crime Thriller,0.760503
4409,Charlie's Angels: Full Throttle (2003),Action Adventure Comedy Crime Thriller,0.760503
7409,Machete (2010),Action Adventure Comedy Crime Thriller,0.760503
9394,Maximum Ride (2016),Action Adventure Comedy Fantasy Sci-Fi Thriller,0.759670
3608,"Stunt Man, The (1980)",Action Adventure Comedy Drama Romance Thriller,0.753983


### Observation

The content-based recommendation system was implemented using TF-IDF representation of movie genres and cosine similarity.  
Two types of recommendations were generated:

1. **Movie-to-movie recommendation**, where similar movies were suggested for a selected title.
2. **User-profile recommendation**, where a user preference vector was created using the weighted average of genre vectors of previously rated movies.

The output shows that the model is able to recommend movies with closely related genre patterns. However, since the model depends only on genre information, some recommendations may appear less intuitive. This happens because the approach does not consider popularity, storyline, or collaborative user behavior.

In [16]:
# ============================================================
# Task 2: Collaborative Filtering
#
# Step 1:
# Create a user–item rating matrix where rows represent users
# and columns represent movies.
# ============================================================

user_item_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

print("User–Item Matrix Shape:", user_item_matrix.shape)
user_item_matrix.head()

User–Item Matrix Shape: (610, 9724)


movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# ============================================================
# Task 2: Collaborative Filtering
#
# Step 2:
# Compute similarity between users using cosine similarity.
# This helps identify users with similar movie preferences.
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity

# Replace NaN with 0 for similarity computation
user_item_filled = user_item_matrix.fillna(0)

# Compute user-user similarity matrix
user_similarity = cosine_similarity(user_item_filled)

print("User similarity matrix shape:", user_similarity.shape)

User similarity matrix shape: (610, 610)


In [18]:
# ============================================================
# Task 2: Collaborative Filtering
#
# Step 3:
# Convert the similarity matrix into a DataFrame so that
# user similarities can be easily accessed and interpreted.
# ============================================================

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

user_similarity_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.027283,0.059720,0.194395,0.129080,0.128152,0.158744,0.136968,0.064263,0.016875,...,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.093572,0.145321
2,0.027283,1.000000,0.000000,0.003726,0.016614,0.025333,0.027585,0.027257,0.000000,0.067445,...,0.202671,0.016866,0.011997,0.000000,0.000000,0.028429,0.012948,0.046211,0.027565,0.102427
3,0.059720,0.000000,1.000000,0.002251,0.005020,0.003936,0.000000,0.004941,0.000000,0.000000,...,0.005048,0.004892,0.024992,0.000000,0.010694,0.012993,0.019247,0.021128,0.000000,0.032119
4,0.194395,0.003726,0.002251,1.000000,0.128659,0.088491,0.115120,0.062969,0.011361,0.031163,...,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.032198,0.107683
5,0.129080,0.016614,0.005020,0.128659,1.000000,0.300349,0.108342,0.429075,0.000000,0.030611,...,0.068048,0.418747,0.110148,0.258773,0.148758,0.106435,0.152866,0.135535,0.261232,0.060792


In [19]:
# ============================================================
# Task 2: Collaborative Filtering
#
# Step 4:
# Find users who are most similar to a selected user.
# ============================================================

target_user = 1

similar_users = user_similarity_df[target_user].sort_values(
    ascending=False
)[1:6]

print("Top 5 users similar to User 1:")
similar_users

Top 5 users similar to User 1:


,1
userId,
266,0.357408
313,0.351562
368,0.345127
57,0.345034
91,0.334727


In [20]:
# ============================================================
# Task 2: Collaborative Filtering
#
# Step 5:
# Recommend movies to the target user based on ratings from
# the most similar users.
# ============================================================

# Get top 5 similar users
top_users = similar_users.index

# Movies already rated by target user
target_user_movies = set(ratings[ratings['userId'] == target_user]['movieId'])

# Ratings from similar users
similar_users_ratings = ratings[ratings['userId'].isin(top_users)]

# Compute average rating for each movie from similar users
movie_recommendation_scores = (
    similar_users_ratings.groupby('movieId')['rating']
    .mean()
    .sort_values(ascending=False)
)

# Remove movies already rated by target user
recommended_movie_ids = [
    movie_id for movie_id in movie_recommendation_scores.index
    if movie_id not in target_user_movies
][:10]

# Map movieId to movie title
user_based_recommendations = movies[movies['movieId'].isin(recommended_movie_ids)][
    ['movieId', 'title', 'genres']
]

print("Top 10 recommendations using User-Based Collaborative Filtering:")
user_based_recommendations

Top 10 recommendations using User-Based Collaborative Filtering:


,movieId,title,genres
449,514,"Ref, The (1994)",Comedy
474,541,Blade Runner (1982),Action Sci-Fi Thriller
942,1243,Rosencrantz and Guildenstern Are Dead (1990),Comedy Drama
1646,2195,Dirty Work (1998),Comedy
2279,3022,"General, The (1926)",Comedy War
2283,3030,Yojimbo (1961),Action Adventure
2308,3060,"Commitments, The (1991)",Comedy Drama Musical
2350,3108,"Fisher King, The (1991)",Comedy Drama Fantasy Romance
2449,3262,Twin Peaks: Fire Walk with Me (1992),Crime Drama Mystery Thriller
2498,3334,Key Largo (1948),Crime Drama Film-Noir Thriller


### Observation

User-based collaborative filtering recommends movies by identifying users with similar rating behavior.  
In this implementation, cosine similarity was used to measure similarity between users based on their rating patterns.

The system first identified the most similar users to the target user and then recommended movies that these similar users rated highly but the target user had not watched yet.

This approach captures shared preferences among users and can recommend movies outside the user's previously watched genres. However, it may suffer from sparsity when many ratings are missing.

In [21]:
# ============================================================
# Task 2: Collaborative Filtering
#
# Step 6:
# Compute similarity between items (movies) using cosine similarity.
# This helps recommend movies similar to the ones a user already liked.
# ============================================================

# Transpose user-item matrix so rows become movies
item_user_matrix = user_item_matrix.T.fillna(0)

# Compute item-item similarity
item_similarity = cosine_similarity(item_user_matrix)

# Convert to DataFrame for easier lookup
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=item_user_matrix.index,
    columns=item_user_matrix.index
)

print("Item similarity matrix shape:", item_similarity_df.shape)

Item similarity matrix shape: (9724, 9724)


In [22]:
# ============================================================
# Task 2: Collaborative Filtering
#
# Step 7:
# Select a movie liked by the target user and find similar movies
# using item-item similarity.
# ============================================================

sample_movie_id = 1  # Toy Story (1995)

similar_movies = item_similarity_df[sample_movie_id].sort_values(
    ascending=False
)[1:11]

print("Top 10 movies similar to movieId 1:")
similar_movies

Top 10 movies similar to movieId 1:


,1
movieId,
3114,0.572601
480,0.565637
780,0.564262
260,0.557388
356,0.547096
364,0.541145
1210,0.541089
648,0.538913
1265,0.534169


In [23]:
# ============================================================
# Task 2: Collaborative Filtering
#
# Step 8:
# Convert the recommended movieIds into movie titles
# for better interpretation of the results.
# ============================================================

similar_movie_ids = similar_movies.index

item_based_recommendations = movies[
    movies['movieId'].isin(similar_movie_ids)
][['movieId', 'title', 'genres']]

print("Movies similar to Toy Story (1995):")
item_based_recommendations

Movies similar to Toy Story (1995):


,movieId,title,genres
224,260,Star Wars: Episode IV - A New Hope (1977),Action Adventure Sci-Fi
314,356,Forrest Gump (1994),Comedy Drama Romance War
322,364,"Lion King, The (1994)",Adventure Animation Children Drama Musical IMAX
418,480,Jurassic Park (1993),Action Adventure Sci-Fi Thriller
546,648,Mission: Impossible (1996),Action Adventure Mystery Thriller
615,780,Independence Day (a.k.a. ID4) (1996),Action Adventure Sci-Fi Thriller
911,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action Adventure Sci-Fi
964,1265,Groundhog Day (1993),Comedy Fantasy Romance
969,1270,Back to the Future (1985),Adventure Comedy Sci-Fi
2355,3114,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy


### Observation

Item-based collaborative filtering recommends movies by identifying items that receive similar rating patterns from users.

In this implementation, cosine similarity was calculated between movies based on the user–item rating matrix. For a selected movie, the system identifies other movies that were rated similarly by many users.

The results show that movies similar to *Toy Story (1995)* include films such as *Toy Story 2*, *The Lion King*, and *Back to the Future*. These movies share similar audience preferences and popularity.

Compared to user-based collaborative filtering, item-based methods are often more stable because the similarity between items changes less frequently than user behavior.

In [24]:
# ============================================================
# Task 3: Matrix Factorization
#
# Step 1:
# Fill missing ratings with 0 and convert the user-item matrix
# into a numerical matrix for Singular Value Decomposition.
# ============================================================

matrix_filled = user_item_matrix.fillna(0)

print("Matrix shape for SVD:", matrix_filled.shape)


Matrix shape for SVD: (610, 9724)


In [26]:
# ============================================================
# Task 3: Matrix Factorization
#
# Step 2:
# Apply Singular Value Decomposition (SVD) to the filled
# user-item matrix.
# ============================================================

U, sigma, Vt = np.linalg.svd(matrix_filled, full_matrices=False)

print("U shape:", U.shape)
print("Sigma shape:", sigma.shape)
print("Vt shape:", Vt.shape)

U shape: (610, 610)
Sigma shape: (610,)
Vt shape: (610, 9724)


In [27]:
# ============================================================
# Task 3: Matrix Factorization
#
# Step 3:
# Reduce the matrix to a smaller number of latent factors.
# This helps capture the most important hidden patterns.
# ============================================================

k = 50

U_k = U[:, :k]
sigma_k = np.diag(sigma[:k])
Vt_k = Vt[:k, :]

print("Reduced U shape:", U_k.shape)
print("Reduced Sigma shape:", sigma_k.shape)
print("Reduced Vt shape:", Vt_k.shape)

Reduced U shape: (610, 50)
Reduced Sigma shape: (50, 50)
Reduced Vt shape: (50, 9724)


In [28]:
# ============================================================
# Task 3: Matrix Factorization
#
# Step 4:
# Reconstruct the approximate user-item rating matrix
# using the reduced latent factors.
# ============================================================

predicted_ratings_svd = np.dot(np.dot(U_k, sigma_k), Vt_k)

print("Reconstructed rating matrix shape:", predicted_ratings_svd.shape)


Reconstructed rating matrix shape: (610, 9724)


In [29]:
# ============================================================
# Task 3: Matrix Factorization
#
# Step 5:
# Convert the reconstructed matrix into a DataFrame so that
# predicted ratings can be easily accessed.
# ============================================================

predicted_ratings_df = pd.DataFrame(
    predicted_ratings_svd,
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)

predicted_ratings_df.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,2.181872,0.393674,0.838186,-0.082365,-0.546279,2.521662,-0.887231,-0.025221,0.196969,1.606758,...,-0.024984,-0.021415,-0.028553,-0.028553,-0.024984,-0.028553,-0.024984,-0.024984,-0.024984,-0.058988
2,0.209809,0.004821,0.030742,0.017252,0.183764,-0.060660,0.083306,0.023797,0.048100,-0.151968,...,0.018895,0.016196,0.021594,0.021594,0.018895,0.021594,0.018895,0.018895,0.018895,0.031966
3,0.013394,0.034726,0.050525,0.000200,-0.005577,0.114673,-0.007461,0.000738,0.004747,-0.061284,...,-0.001612,-0.001382,-0.001843,-0.001843,-0.001612,-0.001843,-0.001612,-0.001612,-0.001612,-0.000530
4,2.012072,-0.394882,-0.290386,0.093864,0.123312,0.259765,0.472667,0.035965,0.011293,-0.021983,...,0.001966,0.001685,0.002247,0.002247,0.001966,0.002247,0.001966,0.001966,0.001966,-0.021462
5,1.336714,0.772954,0.064577,0.113880,0.274994,0.584480,0.251048,0.131534,-0.086310,1.035361,...,-0.004407,-0.003778,-0.005037,-0.005037,-0.004407,-0.005037,-0.004407,-0.004407,-0.004407,-0.006099


In [30]:
# ============================================================
# Task 3: Matrix Factorization
#
# Step 6:
# Recommend movies to a target user using predicted ratings
# from the SVD reconstructed matrix.
# ============================================================

target_user = 1

# Movies already rated by the user
rated_movies = ratings[ratings['userId'] == target_user]['movieId'].tolist()

# Get predicted ratings for the user
user_predictions = predicted_ratings_df.loc[target_user]

# Remove movies already rated
unseen_predictions = user_predictions.drop(rated_movies)

# Get top 10 recommended movies
top_recommendations = unseen_predictions.sort_values(ascending=False).head(10)

# Convert movieIds to movie titles
svd_recommendations = movies[movies['movieId'].isin(top_recommendations.index)][
    ['movieId','title','genres']
]

print("Top recommendations using SVD:")
svd_recommendations

Top recommendations using SVD:


,movieId,title,genres
659,858,"Godfather, The (1972)",Crime Drama
793,1036,Die Hard (1988),Action Crime Thriller
922,1221,"Godfather: Part II, The (1974)",Crime Drama
958,1259,Stand by Me (1986),Adventure Drama
1067,1387,Jaws (1975),Action Horror
1445,1968,"Breakfast Club, The (1985)",Comedy Drama
1544,2080,Lady and the Tramp (1955),Animation Children Comedy Romance
1545,2081,"Little Mermaid, The (1989)",Animation Children Comedy Musical Romance
2110,2804,"Christmas Story, A (1983)",Children Comedy
2996,4011,Snatch (2000),Comedy Crime Thriller


### Observation

Matrix factorization using Singular Value Decomposition (SVD) was applied to the user–item rating matrix.  
The technique decomposes the rating matrix into latent factors that capture hidden relationships between users and movies.

By reconstructing the matrix using a reduced number of latent features, the model can estimate ratings for movies that a user has not watched yet.

The recommendations generated using SVD include well-known highly rated movies such as *The Godfather*, *Die Hard*, and *Stand by Me*. This shows that the model successfully captures underlying preference patterns among users.

Compared to basic collaborative filtering, matrix factorization can handle sparse data more effectively and often produces more accurate recommendations.

In [31]:
# ============================================================
# Task 4: Hybrid Recommendation System
#
# Step 1:
# Combine content-based similarity scores and SVD predicted
# ratings to build a simple hybrid recommender.
# ============================================================

target_user = 1

# Get SVD predicted ratings for the target user
svd_scores = predicted_ratings_df.loc[target_user].copy()

# Get content-based similarity scores already computed earlier
content_scores_series = pd.Series(scores, index=movies['movieId'])

# Build a combined dataframe
hybrid_df = pd.DataFrame({
    'movieId': movies['movieId'],
    'content_score': movies['movieId'].map(content_scores_series),
    'svd_score': movies['movieId'].map(svd_scores)
})

# Remove movies already rated by the user
rated_movies = set(ratings[ratings['userId'] == target_user]['movieId'])
hybrid_df = hybrid_df[~hybrid_df['movieId'].isin(rated_movies)]

print("Hybrid score table created successfully")
hybrid_df.head()

Hybrid score table created successfully


,movieId,content_score,svd_score
1,2,0.507291,0.393674
3,4,0.418646,-0.082365
4,5,0.376009,-0.546279
6,7,0.331492,-0.887231
7,8,0.439785,-0.025221


In [32]:
# ============================================================
# Task 4: Hybrid Recommendation System
#
# Step 2:
# Normalize content-based and SVD scores, then combine them
# using a weighted average.
# ============================================================

# Min-max normalization
hybrid_df['content_score_norm'] = (
    (hybrid_df['content_score'] - hybrid_df['content_score'].min()) /
    (hybrid_df['content_score'].max() - hybrid_df['content_score'].min())
)

hybrid_df['svd_score_norm'] = (
    (hybrid_df['svd_score'] - hybrid_df['svd_score'].min()) /
    (hybrid_df['svd_score'].max() - hybrid_df['svd_score'].min())
)

# Weighted hybrid score
hybrid_df['hybrid_score'] = (
    0.4 * hybrid_df['content_score_norm'] +
    0.6 * hybrid_df['svd_score_norm']
)

print("Hybrid scores computed successfully")
hybrid_df[['movieId', 'content_score_norm', 'svd_score_norm', 'hybrid_score']].head()

Hybrid scores computed successfully


,movieId,content_score_norm,svd_score_norm,hybrid_score
1,2,0.641307,0.386845,0.488630
3,4,0.529245,0.306110,0.395364
4,5,0.475344,0.227432,0.326597
6,7,0.419066,0.169608,0.269391
7,8,0.555967,0.315802,0.411868


In [33]:
# ============================================================
# Task 4: Hybrid Recommendation System
#
# Step 3:
# Recommend top movies based on the final hybrid score.
# ============================================================

top_hybrid = hybrid_df.sort_values(
    by='hybrid_score', ascending=False
).head(10)

hybrid_recommendations = movies[
    movies['movieId'].isin(top_hybrid['movieId'])
][['movieId', 'title', 'genres']]

print("Top recommendations using Hybrid Recommender:")
hybrid_recommendations

Top recommendations using Hybrid Recommender:


,movieId,title,genres
337,380,True Lies (1994),Action Adventure Comedy Romance Thriller
793,1036,Die Hard (1988),Action Crime Thriller
858,1129,Escape from New York (1981),Action Adventure Sci-Fi Thriller
902,1200,Aliens (1986),Action Adventure Horror Sci-Fi
922,1221,"Godfather: Part II, The (1974)",Crime Drama
958,1259,Stand by Me (1986),Adventure Drama
1057,1374,Star Trek II: The Wrath of Khan (1982),Action Adventure Sci-Fi Thriller
1158,1527,"Fifth Element, The (1997)",Action Adventure Comedy Sci-Fi
1211,1610,"Hunt for Red October, The (1990)",Action Adventure Thriller
1445,1968,"Breakfast Club, The (1985)",Comedy Drama


### Observation

The hybrid recommendation system combines two different approaches: content-based filtering and matrix factorization.  
Content-based filtering captures similarity between movies using genre information, while SVD captures hidden user–movie preference patterns from the rating matrix.

To combine both models, the scores were normalized and merged using a weighted average. In this implementation, a higher weight was assigned to the SVD score because collaborative patterns generally provide stronger personalized signals.

The final recommendations include movies such as *Die Hard*, *Aliens*, *Stand by Me*, and *The Breakfast Club*. These recommendations appear more balanced than using only a single model.

This shows that hybrid recommendation systems can improve recommendation quality by reducing the limitations of individual methods.

In [34]:
# ============================================================
# Task 5: Model Evaluation
#
# Step 1:
# Split the rating data into training and testing sets
# for evaluation of recommender system performance.
# ============================================================

train_data, test_data = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

print("Training data shape:", train_data.shape)
print("Testing data shape:", test_data.shape)

Training data shape: (80668, 4)
Testing data shape: (20168, 4)


In [35]:
# ============================================================
# Task 5: Model Evaluation
#
# Step 2:
# Train an SVD model using the Surprise library
# ============================================================

reader = Reader(rating_scale=(0.5, 5.0))

data = Dataset.load_from_df(
    ratings[['userId','movieId','rating']],
    reader
)

trainset, testset = surprise_split(data, test_size=0.2)

svd_model = SVD()

svd_model.fit(trainset)

print("SVD model trained successfully")

SVD model trained successfully


In [36]:
# ============================================================
# Task 5: Model Evaluation
#
# Step 3:
# Evaluate the trained SVD model using RMSE.
# ============================================================

predictions = svd_model.test(testset)

rmse_value = accuracy.rmse(predictions)

print("RMSE value:", rmse_value)

RMSE: 0.8684
RMSE value: 0.8683876082982512


### Evaluation

To evaluate the recommender system, the dataset was divided into training and testing sets.  
An SVD-based matrix factorization model was trained using the Surprise library.

The model was evaluated using **Root Mean Squared Error (RMSE)**, which measures the difference between predicted ratings and actual ratings.

The obtained RMSE value was approximately **0.868**, indicating that the predicted ratings are reasonably close to the true ratings.

Lower RMSE values indicate better prediction accuracy. This result suggests that the matrix factorization model captures user preferences effectively for the MovieLens dataset.

### Explainability in Recommendation Systems

Explainability helps users understand why a particular item was recommended. In recommender systems, explanations improve user trust and transparency.

In this assignment, recommendations were generated using multiple techniques such as content-based filtering, collaborative filtering, matrix factorization, and a hybrid model.

Possible explanations for recommendations include:

- **Content-Based Explanation:**  
  Movies are recommended because they share similar genres with movies the user previously liked.

- **User-Based Collaborative Explanation:**  
  Movies are recommended because other users with similar rating behavior liked them.

- **Item-Based Collaborative Explanation:**  
  Movies are recommended because they are similar to movies that the user has already rated highly.

- **Matrix Factorization Explanation:**  
  The system learns hidden patterns in user preferences and movie characteristics to predict unseen ratings.

Providing explanations alongside recommendations can improve the usability and interpretability of recommendation systems.

### Conclusion

In this assignment, several recommendation system techniques were implemented using the MovieLens dataset.

The following approaches were explored:

- Content-Based Filtering using TF-IDF similarity
- User-Based Collaborative Filtering
- Item-Based Collaborative Filtering
- Matrix Factorization using SVD
- Hybrid Recommendation System
- Model Evaluation using RMSE

Each approach has its advantages and limitations. Content-based methods rely on item features, while collaborative filtering leverages user behavior. Matrix factorization captures hidden relationships in the data and often provides better performance. Hybrid approaches combine multiple techniques to improve recommendation quality.

Overall, recommender systems play a critical role in modern digital platforms by helping users discover relevant content efficiently.